# Local Jupyter Version & Setup Guide
This notebook has been modified for local execution. Follow the instructions below to set up your virtual environment, install all dependencies, link the kernel, and run the pipeline.

## Environment Setup & Dependencies Installation

Run the following commands in your terminal (Command Prompt or PowerShell) inside your project directory:

```bash
# 1. Create a virtual environment named 'amazon_env'
python -m venv amazon_env

# 2. Activate the virtual environment
# For Windows Command Prompt:
amazon_env\Scripts\activate
# For Windows PowerShell:
.\amazon_env\Scripts\activate
# For macOS/Linux:
source amazon_env/bin/activate

# 3. Upgrade pip to the latest version
python -m pip install --upgrade pip

# 4. Install PyTorch with CUDA 12.1 acceleration
pip install torch torchvision torchaudio --index-url [https://download.pytorch.org/whl/cu121](https://download.pytorch.org/whl/cu121)

# 5. Install basic pipeline dependencies, scikit-learn, and UI tools
pip install transformers scikit-learn tqdm ipykernel

# 6. Install essential bitsandbytes + accelerate optimization backends
# (REQUIRED to prevent Windows Virtual Memory / Paging File Error 1455 when loading Qwen)
pip install accelerate bitsandbytes

# 7. Register the environment as a selectable Jupyter Notebook kernel
python -m ipykernel install --user --name=amazon_env --display-name "Python (amazon_env)"

## Running the Notebook

1. Open Jupyter Notebook / JupyterLab or launch this file inside VS Code.
2. Click on the kernel selector tool in the top-right corner of your workspace.
3. Select **"Python (amazon_env)"** from the list to bind the execution to your newly created virtual environment.
4. Paste your HuggingFace User Access Token into **configuration cell 3** to authenticate with the repository server.
5. Run the cells sequentially.

# Amazon Review Summarizer — Multi-Model Benchmark

Generates and **persistently stores** summaries from 5 abstractive models + 1 extractive baseline + Gemini Flash silver references, across 400 products.

**Models:**
- `facebook/bart-large-cnn`
- `sshleifer/distilbart-cnn-12-6`
- `google/pegasus-cnn_dailymail`
- `philschmid/bart-large-cnn-samsum`
- `Falconsai/text_summarization`
- Extractive TF-IDF baseline
- Gemini 1.5 Flash (silver references + its own summaries)

**Storage:** All outputs saved locally so nothing is lost if the session disconnects.


## 1. Mount Google Drive

All outputs are saved here so they survive session disconnects.

In [1]:
import os
from pathlib import Path

# Define the exact models array so it exists in this kernel session
MODELS = [
    'facebook/bart-large-cnn',
    'sshleifer/distilbart-cnn-12-6',
    'google/pegasus-cnn_dailymail',
    'philschmid/bart-large-cnn-samsum',
    'Falconsai/text_summarization',
]

# Set base output directory
SUMMARIES_DIR = Path.cwd() / "outputs"
SUMMARIES_DIR.mkdir(exist_ok=True)

# Create a dedicated subfolder for each model automatically
for model_id in MODELS + ['extractive_tfidf', 'qwen-silver-ref']:
    safe_folder_name = model_id.replace('/', '_')
    model_folder = SUMMARIES_DIR / safe_folder_name
    model_folder.mkdir(exist_ok=True)

print(f"All model subfolders verified and initialized under:\n--> {SUMMARIES_DIR.resolve()}")


All model subfolders verified and initialized under:
--> D:\PPGI\TFMC\Projeto_Final\outputs


## 2. Imports

In [2]:
import os
os.environ["HF_HOME"] = r"D:\PPGI\TFMC\Projeto_Final\.hf_cache"
os.environ["HF_HUB_CACHE"] = r"D:\PPGI\TFMC\Projeto_Final\.hf_cache\hub"

import warnings
warnings.filterwarnings('ignore')

import re, math, json, os, time
import nltk
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

import torch
from datasets import load_dataset
from transformers import pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('vader_lexicon', quiet=True)

DEVICE = 0 if torch.cuda.is_available() else -1
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Using device: {"GPU" if DEVICE == 0 else "CPU"}')


PyTorch 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce GTX 1660 Ti
Using device: GPU


## 3. Configuration

## Credentials

Enter your HuggingFace token below. This is required to access the dataset.
Get a free token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) — read access is sufficient.

> Your token is never stored to disk — it is only held in memory for the duration of this session.

In [ ]:
import getpass, os

# -- HuggingFace Token --------------------------------------------------------
# Prompts for input without echoing the value to the screen.
# Leave blank and press Enter to skip (only works if the dataset is public).
_hf_token = getpass.getpass("HuggingFace token (input hidden): ").strip()

if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    from huggingface_hub import login
    login(token=_hf_token)
    print("[OK] HuggingFace token accepted.")
else:
    os.environ["HF_TOKEN"] = ""
    print("[warn] No token entered — proceeding without authentication.")
    print("       This will fail if the dataset is private or gated.")


In [3]:
import os
from pathlib import Path

# -- Environment --------------------------------------------------------------─
notebook_dir = os.getcwd()
os.environ["HF_HOME"] = os.path.join(notebook_dir, ".hf_cache")
os.environ["HF_HUB_CACHE"] = os.path.join(notebook_dir, ".hf_cache", "hub")

# -- Dataset ------------------------------------------------------------------─
REPO_ID = 'BarbaDLuca/amazon-reviews-2023-with-asin'
CATEGORY = 'Toys_and_Games'

# -- Product filtering --------------------------------------------------------─
MIN_REVIEWS_PER_PRODUCT = 10
MAX_PRODUCTS = 400

# -- Extractive settings ------------------------------------------------------─
EXTRACTIVE_SENTENCES = 3

# -- Abstractive settings ------------------------------------------------------
MAX_INPUT_CHARS = 4000
SUMMARY_MIN_LEN = 60
SUMMARY_MAX_LEN = 180

# -- Qwen Silver Reference Model ----------------------------------------------
QWEN_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'


# -- Models to benchmark ------------------------------------------------------─
MODELS = [
    'facebook/bart-large-cnn',
    'sshleifer/distilbart-cnn-12-6',
    'google/pegasus-cnn_dailymail',
    'philschmid/bart-large-cnn-samsum',
    'Falconsai/text_summarization',
]

# -- Output paths (local) ------------------------------------------------------
BASE_DIR = os.path.join(os.getcwd(), 'outputs')
SUMMARIES_DIR = Path(BASE_DIR)
SILVER_JSON = os.path.join(BASE_DIR, 'silver_references.json')
RESULTS_CSV = os.path.join(BASE_DIR, 'summaries.csv')
EVAL_CSV = os.path.join(BASE_DIR, 'evaluation_results.csv')

os.makedirs(BASE_DIR, exist_ok=True)

print('Config ready.')
print(f'Output dir : {BASE_DIR}')
print(f'HF cache : {os.environ["HF_HOME"]}')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Config ready.
Output dir : D:\PPGI\TFMC\Projeto_Final\outputs
HF cache : D:\PPGI\TFMC\Projeto_Final\.hf_cache


## 4. Load Dataset & Select Products

In [8]:
import random, json, os

PRODUCTS_JSON = os.path.join(BASE_DIR, 'products.json')

if os.path.exists(PRODUCTS_JSON):
    with open(PRODUCTS_JSON, encoding='utf-8') as f:
        products = json.load(f)
    print(f'Loaded {len(products)} saved products from disk — skipping sampling.')
else:
    print(f'Loading category: {CATEGORY or "ALL"} ...')
    load_kwargs = dict(split='train')
    if CATEGORY:
        load_kwargs['data_dir'] = CATEGORY

    ds = load_dataset(REPO_ID, streaming=True, **load_kwargs)

    random.seed(123)
    candidate_products = set()
    target_pool = max(MAX_PRODUCTS * 5, 500)

    for row in ds:
        asin = row.get('parent_asin')
        if asin:
            candidate_products.add(asin)
        if len(candidate_products) >= target_pool:
            break

    products = sorted(random.sample(
        list(candidate_products),
        min(MAX_PRODUCTS, len(candidate_products))
    ))

    with open(PRODUCTS_JSON, 'w', encoding='utf-8') as f:
        json.dump(products, f, indent=2)
    print(f'Selected and saved {len(products)} products (seed=123)')

# Reload stream for review collection
load_kwargs = dict(split='train')
if CATEGORY:
    load_kwargs['data_dir'] = CATEGORY
ds = load_dataset(REPO_ID, streaming=True, **load_kwargs)
print(f'Stream ready. {len(products)} products selected.')

Loaded 400 saved products from disk — skipping sampling.
Stream ready. 400 products selected.


## 5. Collect Reviews

In [9]:
import json
from tqdm.notebook import tqdm

REVIEWS_JSON = os.path.join(BASE_DIR, 'product_reviews.json')

if os.path.exists(REVIEWS_JSON):
    with open(REVIEWS_JSON, encoding='utf-8') as f:
        product_reviews = json.load(f)
    collected = sum(len(v) for v in product_reviews.values())
    counts = [len(product_reviews[a]) for a in products]
    print(f'Loaded {collected} reviews across {len(product_reviews)} products from disk.')
    print(f'Reviews per product — min: {min(counts)}, max: {max(counts)}, mean: {sum(counts)/len(counts):.1f}')

else:
    MAX_REVIEWS_PER_PRODUCT = 50

    print("Processing streaming dataset using clean Python loops with progress bar...")
    product_set = set(products)
    product_reviews = {asin: [] for asin in products}
    collected = 0

    with tqdm(desc="Scanning Amazon Dataset", unit=" rows") as pbar:
        for row in ds:
            pbar.update(1)
            asin = row.get('parent_asin')
            if asin in product_set:
                if len(product_reviews[asin]) < MAX_REVIEWS_PER_PRODUCT:
                    product_reviews[asin].append({
                        'text': str(row.get('text', '')).strip(),
                        'rating': int(row.get('rating', 0)) if row.get('rating') is not None else 0
                    })
                    collected += 1
                    pbar.set_postfix_str(f"Matches: {collected}", refresh=False)

    counts = [len(product_reviews[a]) for a in products]
    print(f'\nFinished! Collected {collected} reviews across {len(products)} products.')
    print(f'Reviews per product — min: {min(counts)}, max: {max(counts)}, mean: {sum(counts)/len(counts):.1f}')

    with open(REVIEWS_JSON, 'w', encoding='utf-8') as f:
        json.dump(product_reviews, f, ensure_ascii=False)
    print(f'Saved to {REVIEWS_JSON}')

Loaded 11007 reviews across 400 products from disk.
Reviews per product — min: 1, max: 50, mean: 27.5


## 6. Storage Helpers

Each product's summaries are saved as an individual JSON file in `summaries/` on Drive.
This means if the session disconnects mid-run, already-completed products are preserved
and the run can resume from where it left off.

In [10]:
import os
import json
from pathlib import Path

def product_path(asin: str, model_id: str) -> Path:
    """Path to the JSON file storing summaries for one product under a specific model's folder."""
    # Convert slashes to underscores to match the initialized folder names
    safe_model_name = model_id.replace('/', '_')
    return SUMMARIES_DIR / safe_model_name / f'{asin}.json'


def load_product(asin: str, model_id: str) -> dict:
    """Load existing model summaries, handling encodings, bad characters, and corruption safely."""
    path = product_path(asin, model_id)
    if path.exists():
        try:
            # Try loading as clean UTF-8
            with open(path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except (json.JSONDecodeError, UnicodeDecodeError):
            # If it's corrupted or saved in old CP1252/Windows format, catch it
            try:
                # Fallback to reading it with cp1252 so it doesn't crash
                with open(path, 'r', encoding='cp1252') as f:
                    return json.load(f)
            except Exception:
                print(f"[warn] Warning: Resetting unreadable/corrupted file for {asin} under {model_id}.")
                return {} # Return empty dict to cleanly overwrite it next save
    return {}


def save_product(asin: str, model_id: str, data: dict):
    """Persist summaries for a product to disk safely, building subdirectories dynamically."""
    path = product_path(asin, model_id)
    
    # FIXED: Automatically build the parent model folder if it does not exist yet
    path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def already_done(asin: str, model_id: str) -> bool:
    """Check if a model's summaries already exist for this product by checking file existence."""
    return product_path(asin, model_id).exists()


def count_completed(model_id: str) -> int:
    """Count how many products have been completed for a specific model."""
    return sum(1 for a in products if already_done(a, model_id))


print('Storage helpers completely robust for folder-per-model architecture.')

Storage helpers completely robust for folder-per-model architecture.


## 7. Review Group Helper

In [12]:
STAR_RATINGS = [1, 2, 3, 4, 5]

def get_review_groups(asin: str) -> dict:
    reviews = product_reviews[asin]
    return {
        'overall': [r['text'] for r in reviews if r['text']],
        'positive_4_5': [r['text'] for r in reviews if r['rating'] >= 4 and r['text']],
        'negative_1_2': [r['text'] for r in reviews if r['rating'] <= 2 and r['text']],
        **{f'{s}_star': [r['text'] for r in reviews if r['rating'] == s and r['text']]
           for s in STAR_RATINGS}
    }

print('Review group helper ready.')

Review group helper ready.


## 8. Qwen2.5-1.5B — Silver Reference Generation

Runs `Qwen2.5-1.5B-Instruct` locally on the GPU in FP16 to generate silver references for all 400 products.

Results are saved incrementally to `outputs/qwen-silver-ref/` — safe to re-run if interrupted.

In [13]:
import os
import gc
import json
import torch
import numpy as np # FIX: Added to prevent NameError inside select_representative_reviews
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

def select_representative_reviews(texts: list, n: int = 15) -> str:
    if not texts:
        return ''
    if len(texts) <= n:
        return ' '.join(texts)[:MAX_INPUT_CHARS]
    try:
        tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
        matrix = tfidf.fit_transform(texts)
        scores = cosine_similarity(matrix, matrix).mean(axis=1)
        top_idx = np.argsort(scores)[-n:][::-1]
        selected = [texts[i] for i in sorted(top_idx)]
        return ' '.join(selected)[:MAX_INPUT_CHARS]
    except Exception:
        return ' '.join(texts[:n])[:MAX_INPUT_CHARS]

silver_references = {}

# Load cached silver references — with corruption recovery
if os.path.exists(SILVER_JSON):
    try:
        with open(SILVER_JSON, encoding='utf-8') as f:
            silver_references = json.load(f)
        print(f'Loaded {len(silver_references)} cached silver references.')
    except (json.JSONDecodeError, UnicodeDecodeError):
        print('[warn] silver_references.json corrupted — rebuilding from product files...')
        for asin in products:
            data = load_product(asin, 'qwen-silver-ref')
            ref = data.get('overall')
            if ref:
                silver_references[asin] = ref
        with open(SILVER_JSON, 'w', encoding='utf-8') as f:
            json.dump(silver_references, f, indent=2, ensure_ascii=False)
        print(f'Rebuilt {len(silver_references)} references from product files.')

TRACKING_KEY = 'qwen-silver-ref'
pending_asins = [
    a for a in products
    if not already_done(a, TRACKING_KEY)
]
print(f'Offline Reference Engine: {len(products) - len(pending_asins)} already done, {len(pending_asins)} remaining.')

# -- Ultra-Low Memory RAM Loading Strategy (Fix for Paging File Error 1455) --
model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Using a kwargs dictionary forces the backend to bypass strict signature matching 
# while accurately forcing raw FP16 shard loading straight to the GPU device map.
loading_kwargs = {
    "torch_dtype": torch.float16,
    "low_cpu_mem_usage": True,
    "device_map": "auto"
}

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    **loading_kwargs
)
model.eval()
print(f'Qwen loaded in FP16. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

try:
    for asin in tqdm(pending_asins, desc='Generating local references (FP16 GPU)'):

        if asin in silver_references and silver_references[asin]:
            save_product(asin, TRACKING_KEY, {'overall': silver_references[asin]})
            continue

        groups = get_review_groups(asin)
        overall_reviews = groups.get('overall', [])
        selected = select_representative_reviews(overall_reviews, n=10)
        combined_text = selected[:2000]

        messages = [
            {'role': 'system', 'content': 'You are an expert review analyzer.'},
            {'role': 'user', 'content': (
                f'Produce a factual, objective 3-4 sentence summary aggregating '
                f'customer viewpoints based on these reviews:\n\n{combined_text}'
            )}
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = None
        summary_ids = None
        try:
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to('cuda')

            with torch.no_grad():
                summary_ids = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    num_beams=4,
                    length_penalty=2.0,
                    early_stopping=True,
                    no_repeat_ngram_size=3,
                    pad_token_id=tokenizer.eos_token_id,
                )

            input_len = inputs['input_ids'].shape[1]
            text_out = tokenizer.decode(
                summary_ids[0][input_len:], skip_special_tokens=True
            ).strip()

            silver_references[asin] = text_out
            save_product(asin, TRACKING_KEY, {'overall': text_out})

            with open(SILVER_JSON, 'w', encoding='utf-8') as f:
                json.dump(silver_references, f, indent=2, ensure_ascii=False)

        except torch.cuda.OutOfMemoryError:
            print(f' [fail] OOM on {asin} — skipping')
        except Exception as e:
            print(f' [fail] {asin}: {e}')
        finally:
            if inputs is not None:
                del inputs
            if summary_ids is not None:
                del summary_ids
            gc.collect()
            torch.cuda.empty_cache()

finally:
    if 'model' in dir():
        del model
    if 'tokenizer' in dir():
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\nVRAM cleanup complete. Total references: {len(silver_references)}')

Loaded 400 cached silver references.
Offline Reference Engine: 400 already done, 0 remaining.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen loaded in FP16. VRAM used: 3.09 GB


Generating local references (FP16 GPU): 0it [00:00, ?it/s]


VRAM cleanup complete. Total references: 400

## 9. Extractive Summarization (TF-IDF Baseline)

In [14]:
import nltk
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

def select_representative_reviews(texts: list, n: int = 8) -> str:
    """
    Use TF-IDF cosine similarity to select the n most representative reviews
    from the full list, then join them. Ensures all reviews compete for
    inclusion rather than blindly taking the first n.
    """
    if not texts:
        return ''
    if len(texts) <= n:
        return ' '.join(texts)[:MAX_INPUT_CHARS]
    try:
        tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
        matrix = tfidf.fit_transform(texts)
        scores = cosine_similarity(matrix, matrix).mean(axis=1)
        top_idx = np.argsort(scores)[-n:][::-1]
        selected = [texts[i] for i in sorted(top_idx)]
        return ' '.join(selected)[:MAX_INPUT_CHARS]
    except Exception:
        return ' '.join(texts[:n])[:MAX_INPUT_CHARS]


def extractive_summary(texts: list, n: int = EXTRACTIVE_SENTENCES) -> str:
    if not texts:
        return 'No reviews available.'
    sentences = []
    for t in texts:
        sentences.extend(nltk.sent_tokenize(t))
    sentences = [s.strip() for s in sentences if len(s.split()) >= 5]
    if not sentences:
        return ' '.join(texts[0].split()[:50])
    if len(sentences) <= n:
        return ' '.join(sentences)
    try:
        tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
        matrix = tfidf.fit_transform(sentences)
        sim = cosine_similarity(matrix, matrix)
        scores = sim.mean(axis=1)
        top_idx = sorted(np.argsort(scores)[-n:][::-1])
        return ' '.join(sentences[i] for i in top_idx)
    except Exception:
        return ' '.join(sentences[:n])


EXT_KEY = 'extractive_tfidf'

# Calculate queue based on whether the file exists in outputs/extractive_tfidf/
pending = [a for a in products if not already_done(a, EXT_KEY)]
print(f'Extractive: {len(products) - len(pending)} already done, {len(pending)} remaining.')

# Added tqdm progress tracking for consistency with other generation loops
for asin in tqdm(pending, desc='Generating Extractive Baselines'):
    groups = get_review_groups(asin)
    group_summaries = {}
    
    for group_label, texts in groups.items():
        if texts:
            group_summaries[group_label] = extractive_summary(texts)

    # FIXED: Save directly as a flat dictionary inside the isolated baseline folder structure
    save_product(asin, EXT_KEY, group_summaries)

print(f'Extractive done. Saved: {count_completed(EXT_KEY)}/{len(products)}')

Extractive: 400 already done, 0 remaining.


Generating Extractive Baselines: 0it [00:00, ?it/s]

Extractive done. Saved: 400/400


## 10. Abstractive Models

Each model is loaded, runs across all 400 products, then unloaded to free VRAM.
Already-completed products are skipped automatically on re-run.

In [15]:
import gc
import os
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm


def run_model(model_id: str):
    # Standardize directory name structure
    clean_model_name = model_id.replace('/', '_')
    print(f"\n-- {model_id} --")

    # Bypass PyTorch security gate
    import transformers.utils.import_utils
    import transformers.modeling_utils
    transformers.utils.import_utils.check_torch_load_is_safe = lambda: None
    transformers.modeling_utils.check_torch_load_is_safe = lambda: None

    # Filter out completed products
    pending = [a for a in products if not already_done(a, clean_model_name)]
    print(f" {len(products) - len(pending)} already done, {len(pending)} remaining.")

    if not pending:
        print(f" [OK] Done. All summaries already exist for {model_id}.")
        return

    print(f" Loading weights...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    is_seq2seq = any(x in model_id.lower() for x in ['bart', 'pegasus', 't5', 'falconsai'])

    # FIX: Enforce torch.float32 strictly for Pegasus to prevent NaN/empty beam loops
    if 'pegasus' in model_id.lower():
        chosen_dtype = torch.float32
    else:
        chosen_dtype = torch.float16

    if is_seq2seq:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_id, torch_dtype=chosen_dtype
        ).to(DEVICE)
    else:
        from transformers import AutoModelForCausalLM
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=chosen_dtype
        ).to(DEVICE)

    model.eval()
    print(f" VRAM after load: {torch.cuda.memory_allocated(DEVICE) / 1024**3:.2f} GB")

    try:
        pbar = tqdm(pending, desc=model_id.split('/')[-1], leave=True)
        for idx, asin in enumerate(pbar):
            groups = get_review_groups(asin)
            group_summaries = {}

            for group_label, texts in groups.items():
                if not texts:
                    continue

                combined_text = select_representative_reviews(texts, n=15)
                
                # Replace tags with spaces to avoid token masking artifacts
                combined_text = combined_text.replace('<n>', ' ').replace('\n', ' ').strip()
                if not combined_text:
                    continue

                inputs = tokenizer(
                    combined_text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=1024
                ).to(DEVICE)

                # Standard generation configurations
                if 'pegasus' in model_id.lower():
                    gen_kwargs = {
                        "max_length": 128,
                        "min_length": 15,
                        "num_beams": 4,
                        "early_stopping": True,
                    }
                else:
                    gen_kwargs = {
                        "max_length": SUMMARY_MAX_LEN,
                        "min_length": SUMMARY_MIN_LEN,
                        "num_beams": 4,
                        "length_penalty": 2.0,
                        "early_stopping": True,
                        "no_repeat_ngram_size": 3,
                    }

                if "bart" in model_id.lower():
                    gen_kwargs["forced_bos_token_id"] = 0

                summary_ids = None
                try:
                    with torch.no_grad():
                        summary_ids = model.generate(**inputs, **gen_kwargs)

                    if is_seq2seq:
                        summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
                    else:
                        input_len = inputs['input_ids'].shape[1]
                        summary_text = tokenizer.decode(
                            summary_ids[0][input_len:], skip_special_tokens=True
                        )
                    
                    summary_text = summary_text.strip()
                    if not summary_text:
                        summary_text = "[Model returned empty sequence]"
                    
                    group_summaries[group_label] = summary_text

                except torch.cuda.OutOfMemoryError:
                    group_summaries[group_label] = '[failed: OOM]'
                    torch.cuda.empty_cache()
                except Exception as e:
                    group_summaries[group_label] = f'[failed: {e}]'

                finally:
                    del inputs
                    if summary_ids is not None:
                        del summary_ids
                    torch.cuda.empty_cache()

            save_product(asin, clean_model_name, group_summaries)

            if idx % 5 == 0:
                gc.collect()
                torch.cuda.empty_cache()

    finally:
        if 'model' in dir():
            del model
        if 'tokenizer' in dir():
            del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        print(f" Cleaning memory for {model_id}...")
        print(f" [OK] Done. Saved: {count_completed(clean_model_name)}/{len(products)}")

# Step 1 — BART models
for model_id in ['facebook/bart-large-cnn', 'sshleifer/distilbart-cnn-12-6']:
    run_model(model_id)


-- facebook/bart-large-cnn --
 400 already done, 0 remaining.
 [OK] Done. All summaries already exist for facebook/bart-large-cnn.

-- sshleifer/distilbart-cnn-12-6 --
 400 already done, 0 remaining.
 [OK] Done. All summaries already exist for sshleifer/distilbart-cnn-12-6.


## 11. Consolidate to CSV

Flattens all per-product JSON files into a single `summaries.csv` for easy loading into interface.

In [16]:
all_rows = []
# Include tracking keys and extractive keys along with your abstractive models list
all_model_keys = ['extractive_tfidf', 'qwen-silver-ref'] + MODELS

print("Consolidating summaries from isolated folder architecture...")

for model_key in all_model_keys:
    safe_folder = model_key.replace('/', '_')
    folder_path = os.path.join(SUMMARIES_DIR, safe_folder)
    
    if not os.path.exists(folder_path):
        print(f"[warn] Skipping {model_key}: Folder does not exist yet.")
        continue
    
    # Read files directly from each model's specific folder
    files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
    print(f" - Reading {len(files)} files from {model_key}...")
    
    for filename in files:
        asin = filename.replace('.json', '')
        
        # Use our updated safe loader helper
        group_dict = load_product(asin, model_key)
        
        if not group_dict or not isinstance(group_dict, dict):
            continue
        
        # Extract summaries per rating group tier
        for group_label, summary_text in group_dict.items():
            # Support any legacy nested structures if they were half-written, otherwise take direct string
            if isinstance(summary_text, dict):
                summary_text = summary_text.get('overall', '') or list(summary_text.values())[0]
            
            # --- AJUSTE EXCLUSIVO PARA O PEGASUS ---
            # Limpa os marcadores <n> para o ROUGE e BERTScore calcularem a nota real do modelo
            summary_clean = str(summary_text).replace('<n>', ' ').replace('  ', ' ').strip()
            
            texts = get_review_groups(asin).get(group_label, [])
            
            all_rows.append({
                'asin': asin,
                'category': CATEGORY or 'ALL',
                'model': model_key,
                'group': group_label,
                'review_count': len(texts),
                'summary': summary_clean, # <--- Usando o texto limpo aqui
            })

results_df = pd.DataFrame(all_rows)
results_df.to_csv(RESULTS_CSV, index=False)

print(f'\n[OK] Success! Consolidated {len(results_df)} total group rows -> {RESULTS_CSV}')
print("\nProducts fully covered per model variant:")
print(results_df.groupby('model')['asin'].nunique().rename('products_covered'))

Consolidating summaries from isolated folder architecture...
 - Reading 400 files from extractive_tfidf...
 - Reading 400 files from qwen-silver-ref...
 - Reading 400 files from facebook/bart-large-cnn...
 - Reading 400 files from sshleifer/distilbart-cnn-12-6...
 - Reading 400 files from google/pegasus-cnn_dailymail...
 - Reading 400 files from philschmid/bart-large-cnn-samsum...
 - Reading 400 files from Falconsai/text_summarization...

[OK] Success! Consolidated 14692 total group rows -> D:\PPGI\TFMC\Projeto_Final\outputs\summaries.csv

Products fully covered per model variant:
model
Falconsai/text_summarization        400
extractive_tfidf                    400
facebook/bart-large-cnn             400
google/pegasus-cnn_dailymail        400
philschmid/bart-large-cnn-samsum    400
qwen-silver-ref                     400
sshleifer/distilbart-cnn-12-6       400
Name: products_covered, dtype: int64


## 12. Evaluation

Metrics:
- **ROUGE-1/2/L** + **BERTScore** — vs Qwen silver reference (overall group only)
- **BARTScore** — fluency + relevance
- **Sentiment alignment** — summary sentiment vs majority review sentiment

In [18]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score
from nltk.sentiment import SentimentIntensityAnalyzer

rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
sia = SentimentIntensityAnalyzer()

summac_model = None
try:
    from summac.model_summac import SummaCZS
    summac_model = SummaCZS(granularity='sentence', model_name='vitc')
    print('SummaC loaded.')
except Exception as e:
    print(f'[warn] SummaC not available: {e}')


def sentiment_alignment(reviews: list, summary: str) -> float:
    scores = [sia.polarity_scores(r)['compound'] for r in reviews]
    maj_pos = sum(1 for s in scores if s > 0) > len(scores) / 2
    sum_pos = sia.polarity_scores(summary)['compound'] > 0
    return 1.0 if maj_pos == sum_pos else 0.0


# -- Split rows: overall group with silver ref vs everything else --------------─
overall_mask = (results_df['group'] == 'overall') & (results_df['asin'].isin(silver_references))
ref_rows = results_df[overall_mask].copy().reset_index(drop=True)
noref_rows = results_df[~overall_mask].copy().reset_index(drop=True)

print(f'Rows with silver reference : {len(ref_rows)}')
print(f'Rows without reference     : {len(noref_rows)}')

# -- BERTScore — single batch (loads roberta-large once) ----------------------─
print(f'\nRunning BERTScore on {len(ref_rows)} rows (single model load)...')
ref_summaries_clean = [
    s if s else '[no summary]'
    for s in ref_rows['summary'].fillna('').str.strip().tolist()
]
ref_references = [silver_references[a] for a in ref_rows['asin']]

_, _, F = bertscore_score(
    ref_summaries_clean, ref_references,
    lang='en', verbose=True, batch_size=32
)
ref_rows['bertscore_f1'] = F.tolist()

# -- ROUGE — fast row-by-row, no model loading --------------------------------─
print('Running ROUGE...')
r1, r2, rL = [], [], []
for _, row in ref_rows.iterrows():
    r = rouge.score(silver_references[row['asin']], row['summary'])
    r1.append(r['rouge1'].fmeasure)
    r2.append(r['rouge2'].fmeasure)
    rL.append(r['rougeL'].fmeasure)
ref_rows['rouge1_f'] = r1
ref_rows['rouge2_f'] = r2
ref_rows['rougeL_f'] = rL

# -- No-reference rows --------------------------------------------------------─
noref_rows['bertscore_f1'] = None
noref_rows['rouge1_f'] = None
noref_rows['rouge2_f'] = None
noref_rows['rougeL_f'] = None

# -- Sentiment alignment + SummaC for all rows --------------------------------─
eval_df = pd.concat([ref_rows, noref_rows]).reset_index(drop=True)

print(f'\nRunning sentiment alignment + SummaC on {len(eval_df)} rows...')
summac_scores, sentiment_scores = [], []

for i, row in eval_df.iterrows():
    if i % 500 == 0:
        print(f'  {i}/{len(eval_df)}...')

    source_reviews = [r['text'] for r in product_reviews[row['asin']] if r['text']]
    source_text = ' '.join(source_reviews)

    if summac_model:
        try:
            summac_scores.append(
                float(summac_model.score([source_text], [row['summary']])['scores'][0])
            )
        except Exception:
            summac_scores.append(None)
    else:
        summac_scores.append(None)

    sentiment_scores.append(sentiment_alignment(source_reviews, row['summary']))

eval_df['summac'] = summac_scores
eval_df['sentiment_alignment'] = sentiment_scores

eval_df.to_csv(EVAL_CSV, index=False)
print(f'\nEvaluation complete -> {EVAL_CSV}')
print(f'Total rows evaluated: {len(eval_df)}')

[warn] SummaC not available: No module named 'summac'
Rows with silver reference : 2800
Rows without reference     : 11892

Running BERTScore on 2800 rows (single model load)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/88 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/88 [00:00<?, ?it/s]

done in 51.25 seconds, 54.63 sentences/sec
Running ROUGE...

Running sentiment alignment + SummaC on 14692 rows...
  0/14692...
  500/14692...
  1000/14692...
  1500/14692...
  2000/14692...
  2500/14692...
  3000/14692...
  3500/14692...
  4000/14692...
  4500/14692...
  5000/14692...
  5500/14692...
  6000/14692...
  6500/14692...
  7000/14692...
  7500/14692...
  8000/14692...
  8500/14692...
  9000/14692...
  9500/14692...
  10000/14692...
  10500/14692...
  11000/14692...
  11500/14692...
  12000/14692...
  12500/14692...
  13000/14692...
  13500/14692...
  14000/14692...
  14500/14692...

Evaluation complete -> D:\PPGI\TFMC\Projeto_Final\outputs\evaluation_results.csv
Total rows evaluated: 14692


## 13. Results — Model Comparison

In [19]:
numeric_cols = ['rouge1_f', 'rouge2_f', 'rougeL_f', 'bertscore_f1',
                'summac', 'sentiment_alignment']

overall_table = (
    eval_df[eval_df['group'] == 'overall']
    .groupby('model')[numeric_cols]
    .mean(numeric_only=True)
    .round(4)
    .sort_values('bertscore_f1', ascending=False)
)

display(Markdown('### Overall group — mean scores per model (400 products)'))
display(overall_table)

all_groups_table = (
    eval_df.groupby('model')[numeric_cols]
    .mean(numeric_only=True)
    .round(4)
    .sort_values('bertscore_f1', ascending=False)
)

display(Markdown('### All groups — mean scores per model'))
display(all_groups_table)

### Overall group — mean scores per model (400 products)

,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sentiment_alignment
model,,,,,
qwen-silver-ref,1.0000,1.0000,1.0000,1.0000,0.9425
philschmid/bart-large-cnn-samsum,0.3139,0.0615,0.1924,0.8727,0.8900
facebook/bart-large-cnn,0.2938,0.0479,0.1743,0.8645,0.9225
extractive_tfidf,0.2554,0.0428,0.1616,0.8609,0.9550
Falconsai/text_summarization,0.3036,0.0500,0.1687,0.8593,0.9575
sshleifer/distilbart-cnn-12-6,0.3006,0.0503,0.1777,0.8593,0.8700
google/pegasus-cnn_dailymail,0.2405,0.0405,0.1578,0.8587,0.8275


### All groups — mean scores per model

,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sentiment_alignment
model,,,,,
qwen-silver-ref,1.0000,1.0000,1.0000,1.0000,0.9425
philschmid/bart-large-cnn-samsum,0.3139,0.0615,0.1924,0.8727,0.7414
facebook/bart-large-cnn,0.2938,0.0479,0.1743,0.8645,0.8081
extractive_tfidf,0.2554,0.0428,0.1616,0.8609,0.8069
Falconsai/text_summarization,0.3036,0.0500,0.1687,0.8593,0.8060
sshleifer/distilbart-cnn-12-6,0.3006,0.0503,0.1777,0.8593,0.7695
google/pegasus-cnn_dailymail,0.2405,0.0405,0.1578,0.8587,0.7116


## 14. Display Summaries Side-by-Side

In [20]:
def display_product_comparison(asin: str, group: str = 'overall'):
    rows = results_df[(results_df['asin'] == asin) & (results_df['group'] == group)]
    ref = silver_references.get(asin, '_No silver reference available_')

    display(Markdown(f'---\n## Product `{asin}` — group: `{group}`'))
    display(Markdown(f'**Silver reference (Qwen2.5-1.5B):**\n{ref}'))

    for _, row in rows.iterrows():
        display(Markdown(
            f'**{row["model"]}** ({row["review_count"]} reviews):\n{row["summary"]}'
        ))


# Show first 3 products as a sample
for asin in products[:3]:
    display_product_comparison(asin, group='overall')

---
## Product `1338106457` — group: `overall`

**Silver reference (Qwen2.5-1.5B):**
The customer reviews indicate that the soap-making kit is well-received by children and adults alike, with many praising its ease of use, variety of scents and colors, and ability to make multiple soaps at once. However, some users note that the kit is basic and may not be suitable for more advanced soap makers, and that the molds and fragrances can be subtle. Users also mention the need for adult supervision due to the hot nature of the process and the potential for the soap to stick to the mold. Despite these challenges, the kits are generally recommended as a fun and educational craft activity for children.

**extractive_tfidf** (50 reviews):
Just soak the stirring spoon and measuring glass in water overnight and the hardened soap dissolves<br />-The projects are cute and my kid loved them<br /><br />Cons:<br />-Melted soap is extremely hot and hardens extremely quickly, so I had to do all of the heating and pouring<br />-If using the microwave method, it's incredibly difficult to keep the soap from bubbling while you heat it, which makes the finished result look cloudy. My daughter loved making this soap and was super easy. I bought this as a birthday gift for my 7 year old niece and she loved it.

**qwen-silver-ref** (50 reviews):
The customer reviews indicate that the soap-making kit is well-received by children and adults alike, with many praising its ease of use, variety of scents and colors, and ability to make multiple soaps at once. However, some users note that the kit is basic and may not be suitable for more advanced soap makers, and that the molds and fragrances can be subtle. Users also mention the need for adult supervision due to the hot nature of the process and the potential for the soap to stick to the mold. Despite these challenges, the kits are generally recommended as a fun and educational craft activity for children.

**facebook/bart-large-cnn** (50 reviews):
Bought for our niece and she absolutely loved it! Pros: It's a quick activity, so doesn't take hours and hours. Just soak the stirring spoon and measuring glass in water overnight and the hardened soap dissolves. Cons: Melted soap is extremely hot and hardens extremely quickly, so I had to do all of the heating and pouring.

...

**facebook/bart-large-cnn** (50 reviews):
Don't have to sharpen, just twist. Great for traveling or in the car, and seem to be tough (crayons don't break) These are nothing but crayons, which you can't sharpen or adjust the clarity of by turning the angle of the &#34;crayon.

**sshleifer/distilbart-cnn-12-6** (50 reviews):
Don't have to sharpen, just twist . Easy to hold and don't break like regular crayons . Great for the price paid easy to hold . Don't make a mess, great for traveling or in the car, and seem to be tough . Save your money; just buy a crayola pack .

**google/pegasus-cnn_dailymail** (50 reviews):
Great price for high quality crayons! They don't break like other crayons and self sharpen with a twist . Save your money; just buy a crayola pack of 96!

**philschmid/bart-large-cnn-samsum** (50 reviews):
The best crayons for kids are easy to hold and they don't break. They don't have to sharpen, just twist. They're great for traveling or in the car and seem to be tough. The quality isn't as good as cheap ones, so buy a crayola pack of 96.

**Falconsai/text_summarization** (50 reviews):
. They don't break like other crayons and self sharpen with a twist. Great price for high quality crayons! Love them I bought these for my four year old grand daughter and she loves them. They come in great sort of colors also. Kids love them! And they don’t really work that great and the quality isn't as good as crayons. Save your money; just buy a crayola pack of 96! My 3 year olds grip on crayons normally snaps them in half

## 15. Query a Specific Product & Group

In [21]:
QUERY_ASIN = products[0]
QUERY_GROUP = 'overall' # 'overall', 'positive_4_5', 'negative_1_2', '1_star'..'5_star'

if QUERY_ASIN in results_df['asin'].values:
    display_product_comparison(QUERY_ASIN, QUERY_GROUP)
else:
    print(f'{QUERY_ASIN} not found.')
    print(f'Available: {list(results_df["asin"].unique()[:10])}...')

---
## Product `1338106457` — group: `overall`

**Silver reference (Qwen2.5-1.5B):**
The customer reviews indicate that the soap-making kit is well-received by children and adults alike, with many praising its ease of use, variety of scents and colors, and ability to make multiple soaps at once. However, some users note that the kit is basic and may not be suitable for more advanced soap makers, and that the molds and fragrances can be subtle. Users also mention the need for adult supervision due to the hot nature of the process and the potential for the soap to stick to the mold. Despite these challenges, the kits are generally recommended as a fun and educational craft activity for children.

**extractive_tfidf** (50 reviews):
Just soak the stirring spoon and measuring glass in water overnight and the hardened soap dissolves<br />-The projects are cute and my kid loved them<br /><br />Cons:<br />-Melted soap is extremely hot and hardens extremely quickly, so I had to do all of the heating and pouring<br />-If using the microwave method, it's incredibly difficult to keep the soap from bubbling while you heat it, which makes the finished result look cloudy. My daughter loved making this soap and was super easy. I bought this as a birthday gift for my 7 year old niece and she loved it.

**qwen-silver-ref** (50 reviews):
The customer reviews indicate that the soap-making kit is well-received by children and adults alike, with many praising its ease of use, variety of scents and colors, and ability to make multiple soaps at once. However, some users note that the kit is basic and may not be suitable for more advanced soap makers, and that the molds and fragrances can be subtle. Users also mention the need for adult supervision due to the hot nature of the process and the potential for the soap to stick to the mold. Despite these challenges, the kits are generally recommended as a fun and educational craft activity for children.

**facebook/bart-large-cnn** (50 reviews):
Bought for our niece and she absolutely loved it! Pros: It's a quick activity, so doesn't take hours and hours. Just soak the stirring spoon and measuring glass in water overnight and the hardened soap dissolves. Cons: Melted soap is extremely hot and hardens extremely quickly, so I had to do all of the heating and pouring.

**sshleifer/distilbart-cnn-12-6** (50 reviews):
Bought for our niece and she absolutely loved it! It's a quick activity, so doesn't take hours and hours . The project took about 1.5 hours (including hardening time for the soaps, which is 30 min to 1 hr during which you cannot use the soap mold). My daughter loved making this soap and was super easy .

**google/pegasus-cnn_dailymail** (50 reviews):
Reviewer: My daughter loved making the soaps and they turned out pretty much like they look on hand . The purple is EXTREMELY dark, it almost looks black (my kid thinks this is a "pro," not a "con," because she put a lot of glitter in the soap, and she thinks the cup looks like a galaxy filled with stars. It looks cool, but if you and your kid are expecting a bright purple, you'll be disappointed. The time between the hotmelt and the microwave is 10 - 15 seconds .

**philschmid/bart-large-cnn-samsum** (50 reviews):
Bought for their niece and she loved it. The project took about 1.5 hours, including hardening time for the soaps. My daughter loves walking soap-making videos on YouTube, so this was a great gift for her and a fun craft. My 6-year-old son enjoyed it too.

**Falconsai/text_summarization** (50 reviews):
and easy to use and works well as a gift. Loved Easy to use. The soap bars are just like regular soap and they smell good the kids had fun making itbr />-The soaps were cute and my niece loved it!! I bought this for my 7 year old niece and she absolutely loved it. It was super easy to do (surprisingly easy, in fact) and they turned out beautiful. I still think this is a great kit, and the soaps turned out nice. I am glad that